# Recherche du meilleur seuil de décision 

Via les notebooks d'exploration et de comparaisons des modèles, nous sommes arrivés à la conclusion qu'ici c'est un XGBoost qui permet d'obtenir les meilleurs performances pour répondre à notre problématique. Au vu des résultats obtenus sur les différentes configs testées, nous ne pensons pas pouvoir améliorer de beaucoup le modèle en tant que tel. Ce notebook à donc juste pour objectif de chercher à optimiser le seuil de décision.

In [ ]:
import mlflow
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

from base_commune.outils import (
    RANDOM_STATE,
    charger_donnees,
    configurer_mlflow,
    decouper_donnees,
    exporter_runs,
    tags_reproductibilite,
)

creer_le_run = False  # encore une fois sinon jai encre le mm problème

In [2]:
df_brut = charger_donnees()
X_train, X_test, y_train, y_test = decouper_donnees(df_brut)
print(X_train.shape, X_test.shape)

(8000, 10) (2000, 10)


In [3]:
vars_num = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "EstimatedSalary",
            "HasCrCard", "IsActiveMember"]
vars_cate = ["Geography", "Gender"]


def construire_xgb(n_estimators=100, max_depth=6, learning_rate=0.3, scale_pos_weight=1):
    preprocesseur = ColumnTransformer([
        ("cate", OneHotEncoder(handle_unknown="ignore"), vars_cate),
        ("num", "passthrough", vars_num),
    ], remainder="drop")
    return Pipeline([
        ("preprocesseur", preprocesseur),
        ("xgb", XGBClassifier(n_estimators=n_estimators, max_depth=max_depth,
                              learning_rate=learning_rate, scale_pos_weight=scale_pos_weight,
                              random_state=RANDOM_STATE, n_jobs=-1)),
    ])


hyperparametres_retenus = {"n_estimators": 300, "max_depth": 3,
                           "learning_rate": 0.05, "scale_pos_weight": 1}
modele = construire_xgb(**hyperparametres_retenus)

In [4]:
decoupage_en_plis = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
probabilites_hors_pli = cross_val_predict(
    modele, X_train, y_train, cv=decoupage_en_plis, method="predict_proba")[:, 1]
precisions, rappels, seuils = precision_recall_curve(y_train, probabilites_hors_pli)
scores_f1 = 2 * precisions * rappels / (precisions + rappels + 1e-12)

In [5]:
def mesurer_au_seuil(seuil):
    predictions = (probabilites_hors_pli >= seuil).astype(int)
    return (precision_score(y_train, predictions),
            recall_score(y_train, predictions),
            f1_score(y_train, predictions))

indice_meilleur_f1 = np.argmax(scores_f1[:-1]) # pr trver le seuil qui maximise le F1 (arrondi à 2 décimales)
seuil_f1_max = round(float(seuils[indice_meilleur_f1]), 2)

rappel_minimum = 0.70 # et la pr avoir le plus grand seuil qui atteint le rappel minimal visé
indices_possibles = np.where(rappels[:-1] >= rappel_minimum)[0]
seuil_rappel_minimum = round(float(seuils[indices_possibles[-1]]), 2)

resultats = {
    "seuil 0,5 (défaut)": 0.5,
    "F1 maximal": seuil_f1_max,
    f"rappel >= {rappel_minimum}": seuil_rappel_minimum,
}
mesures = {nom: mesurer_au_seuil(seuil) for nom, seuil in resultats.items()}
for nom, seuil in resultats.items():
    precision, rappel, f1 = mesures[nom]
    print(f"{nom:20s} seuil {seuil:.2f} | précision {precision:.3f} | rappel {rappel:.3f} | F1 {f1:.3f}")

seuil 0,5 (défaut)   seuil 0.50 | précision 0.771 | rappel 0.457 | F1 0.574
F1 maximal           seuil 0.30 | précision 0.615 | rappel 0.650 | F1 0.632
rappel >= 0.7        seuil 0.25 | précision 0.555 | rappel 0.701 | F1 0.619


In [6]:
# Pour enregistrer dans MLFlow 

configurer_mlflow("churn_bancaire")

if creer_le_run:
    with mlflow.start_run(run_name="xgb_seuil_decision"):
        mlflow.set_tags({"model_family": "xgboost", "author": "roman",
                         "stage": "seuil_decision", **tags_reproductibilite()})
        mlflow.log_params({
            **hyperparametres_retenus,
            "cv_folds": 5,
            "random_state": RANDOM_STATE,
            "test_size": 0.2,
            "n_train": len(X_train),
            "threshold_rule_retained": "f1_max",
            "threshold_f1_max": seuil_f1_max,
            "threshold_recall_rule": seuil_rappel_minimum,
            "recall_target": rappel_minimum,
        })
        for suffixe, nom in [("at_0_5", "seuil 0,5 (défaut)"), ("f1_max", "F1 maximal"),
                             ("recall_rule", f"rappel >= {rappel_minimum}")]:
            precision, rappel, f1 = mesures[nom]
            mlflow.log_metrics({f"precision_{suffixe}": precision,
                                f"recall_{suffixe}": rappel,
                                f"f1_{suffixe}": f1})

In [7]:
chemin_du_fichier_csv = exporter_runs("churn_bancaire", "roman")
print(chemin_du_fichier_csv)

C:\Users\33698\Desktop\Cours\Master\M2 DS SIAD\projet_mlops\mlops_roman_delbare\reports\runs_roman.csv


# Conclusion

On voit qu'en cherchant un seuil de décision qui maximise le F1 on arrive à un rappel de 66% (2/3 des churners detectés) avec une précision correcte (61% des clients détectés comme churners le sont effectivement). 
Si on cherche à augmenter d'avantage le rappel, par exemple en fixant rappel min = 70%, la perte marginale de précision devient trop important (dans ce cas par exemple, +4 points de rappels = 6 points de précision). Si on rapporte ces "comparaisons marrginales" à notre proportion de churner/non churner on vq d'un pdv métier chercher à augmenter d'avantage le rappel n'est pas "rentable" : si on faisait tourner le modèle en prod dans un cas réel on contacterait des centaines de clients en plus pour aller capter quelques dizaines de churnes.

On va donc conserver un seuil à 0.3, qui optimise le F1 et donne ici un mix rappel/précision satisfaisant au vu de notre problématique